# Unsupervised Learning

This notebook introduces the unsupervised learning setting and surveys the algorithms covered in the `ml_002_` series.

## Learning Objectives
By the end of this notebook, you will be able to:
1. **Distinguish** unsupervised from supervised learning, and identify the absence of labels as the defining characteristic.
2. **Categorise** unsupervised tasks: clustering, density estimation, dimensionality reduction, and source separation.
3. **Map** each algorithm in this series to its task category and its core mathematical objective.
4. **Select** the appropriate method given a problem description and dataset properties.
5. **Recognise** the common thread of latent variables across GMM, Factor Analysis, ICA, and PCA.

> **Prerequisite**: Supervised learning (`ml_001_` series), multivariate calculus, probability and the Gaussian distribution.

## 1. What Is Unsupervised Learning?

In **supervised learning** we are given labelled pairs $\{(x^{(i)}, y^{(i)})\}$ and we learn a mapping $x \mapsto y$.

In **unsupervised learning** we are given only the inputs $\{x^{(1)}, \ldots, x^{(m)}\}$ — **no labels $y$** — and we try to discover **hidden structure** in the data.

### Why it matters
- Labels are expensive: labelling 1 million images costs far more than collecting them.
- Much of the world's data is unlabelled (web text, sensor streams, genomic sequences).
- Understanding the structure of $p(x)$ is often the goal itself (data compression, anomaly detection, generative modelling).

### Formal setting

| | Supervised | Unsupervised |
|---|---|---|
| **Training data** | $(x^{(i)}, y^{(i)})$ pairs | $x^{(i)}$ only |
| **Goal** | Learn $p(y \mid x)$ or $f(x)\approx y$ | Learn $p(x)$, structure, or latent $z$ |
| **Evaluation** | Prediction error on held-out $(x,y)$ | Likelihood, reconstruction error, or downstream task |
| **Example tasks** | Regression, classification | Clustering, compression, generation |

## 2. Taxonomy of Unsupervised Tasks

### 2.1 Clustering
**Goal:** Partition $\{x^{(i)}\}$ into $k$ groups (clusters) so that points within a cluster are more similar to each other than to points in other clusters.

- **Hard clustering:** Each point belongs to exactly one cluster.  
  Algorithm: **K-means** (minimises intra-cluster squared distance).
- **Soft clustering / density estimation:** Each point has a probability of belonging to each cluster.  
  Algorithm: **Gaussian Mixture Models** fit via the **EM algorithm**.

### 2.2 Density Estimation
**Goal:** Estimate the probability density $p(x)$ from samples.  
Allows: likelihood evaluation, anomaly detection, sampling synthetic data.

- **Parametric:** Assume $p(x)$ belongs to a family (e.g. mixture of Gaussians).
- **Non-parametric:** Kernel density estimation — no fixed family assumed.

### 2.3 Dimensionality Reduction (Representation Learning)
**Goal:** Find a low-dimensional representation $z \in \mathbb{R}^k$ ($k \ll n$) that captures the most important structure in $x \in \mathbb{R}^n$.

- **Linear, variance-maximising:** **PCA** — top-$k$ eigenvectors of the covariance matrix.
- **Linear, probabilistic:** **Factor Analysis** — $x = \mu + \Lambda z + \varepsilon$; handles $n \gg m$.

### 2.4 Source Separation / Blind Signal Separation
**Goal:** Given mixed signals $x = As$, recover the original independent sources $s$.

- **ICA (Independent Components Analysis):** Learns unmixing matrix $W = A^{-1}$ by maximising statistical independence (non-Gaussianity) of recovered sources.

### 2.5 The Latent Variable Thread

Almost all unsupervised algorithms in this series share a **latent variable** view:

$$x^{(i)} \text{ is observed}, \qquad z^{(i)} \text{ is hidden (latent)}$$

| Algorithm | Latent variable $z$ | How it is inferred |
|---|---|---|
| K-means | Cluster assignment $c^{(i)} \in \{1,\ldots,k\}$ | Hard assignment (argmin distance) |
| GMM/EM | Cluster membership $z \sim \text{Categorical}(\phi)$ | Soft posterior $w_j^{(i)} = p(z=j\mid x^{(i)})$ |
| Factor Analysis | Continuous $z \sim \mathcal{N}(0,I)$ | Gaussian posterior $p(z\mid x)$ |
| PCA | Projection coordinate $z = U_k^T(x-\mu)$ | Deterministic linear projection |
| ICA | Independent source $s = Wx$ | Deterministic linear unmixing |

## 3. Algorithm Survey

### 3.1 K-Means Clustering  (`ml_002_01_1`)

**Objective:** Minimise the **distortion function**
$$J(c, \mu) = \sum_{i=1}^m \|x^{(i)} - \mu_{c^{(i)}}\|^2$$

**Algorithm:** Two-step coordinate descent.
- **Assign:** $c^{(i)} := \arg\min_j \|x^{(i)} - \mu_j\|^2$ (minimise $J$ over $c$, holding $\mu$ fixed)
- **Update:** $\mu_j := \text{mean}\{x^{(i)} : c^{(i)}=j\}$ (minimise $J$ over $\mu$, holding $c$ fixed)

**Properties:** Guaranteed to converge; converges to local minimum; use multiple random restarts.

---

### 3.2 Gaussian Mixture Models  (`ml_002_02_1`)

**Model:** $p(x) = \sum_{j=1}^k \phi_j \, \mathcal{N}(x;\, \mu_j, \Sigma_j)$

**Latent variable:** $z^{(i)} \in \{1,\ldots,k\}$ indicates the mixture component.

**Why GMM over k-means?**
- Soft assignments: each point partially belongs to each cluster.
- Models cluster shape through $\Sigma_j$ (not just centroid distance).
- Provides $p(x)$ — a full density estimate, not just cluster labels.

---

### 3.3 EM Algorithm for GMM  (`ml_002_02_2`)

**Challenge:** Maximising $\ell(\phi, \mu, \Sigma) = \sum_i \log \sum_j \phi_j \mathcal{N}(x^{(i)};\mu_j,\Sigma_j)$ directly is intractable (log of sum).

**EM solution:**
- **E-step:** Compute soft assignments $w_j^{(i)} = p(z^{(i)}=j \mid x^{(i)})$.
- **M-step:** Maximise complete-data log-likelihood using $w_j^{(i)}$ as weights.

K-means is a special case of EM where $w_j^{(i)} \in \{0,1\}$ (hard).

---

### 3.4 Jensen's Inequality  (`ml_002_02_3`)

For a **concave** function $f$ and random variable $X$:
$$f(E[X]) \geq E[f(X)]$$

The log function is concave, so $\log E[X] \geq E[\log X]$.

This inequality is the key tool that proves EM constructs a valid lower bound on the log-likelihood.

---

### 3.5 General EM Framework  (`ml_002_02_4`)

EM maximises the **Evidence Lower BOund (ELBO)**:
$$J(Q, \theta) = \sum_i \sum_z Q_i(z) \log \frac{p(x^{(i)}, z;\theta)}{Q_i(z)} \leq \ell(\theta)$$

The bound is **tight** when $Q_i(z) = p(z \mid x^{(i)}; \theta)$.

EM is **coordinate ascent on $J$**: E-step optimises over $Q$, M-step optimises over $\theta$.

---

### 3.6 Gaussian Conditionals  (`ml_002_03_1`)

If $\begin{bmatrix}x_1\\x_2\end{bmatrix} \sim \mathcal{N}(\mu, \Sigma)$, then:
$$x_1 \mid x_2 \sim \mathcal{N}(\mu_1 + \Sigma_{12}\Sigma_{22}^{-1}(x_2-\mu_2),\; \Sigma_{11} - \Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21})$$

This formula (Schur complement) is the workhorse of the Factor Analysis E-step.

---

### 3.7 Factor Analysis  (`ml_002_03_2`, `ml_002_03_3`)

**Why?** When $m < n$ (fewer data points than dimensions), the sample covariance is singular — full-covariance MLE fails.

**Model:** $z \sim \mathcal{N}(0,I)$, $\;x\mid z \sim \mathcal{N}(\mu + \Lambda z, \Psi)$

**Marginal:** $x \sim \mathcal{N}(\mu,\; \Lambda\Lambda^T + \Psi)$ — low-rank + diagonal covariance.

**Parameters:** $nk + n$ instead of $n^2/2$ — scalable to high dimensions.

Fit via EM; E-step gives posterior $p(z\mid x)$ via Gaussian conditional formula.

---

### 3.8 PCA  (`ml_002_04_1`, `ml_002_04_2`)

**Objective:** Find $k$ orthonormal directions $u_1,\ldots,u_k$ that maximise projected variance:
$$\max_{\|u\|=1} \frac{1}{m}\sum_i (u^T x^{(i)})^2 = u^T \Sigma u$$

**Solution:** $u_j$ = $j$-th eigenvector of $\Sigma$ (with eigenvalue $\lambda_j$).

**Applications:** Data compression, visualisation (project to 2D/3D), noise reduction (discard small-eigenvalue directions), eigenfaces.

**PCA vs Factor Analysis:**
- PCA: implicit equal noise; deterministic projection; no generative model.
- FA: explicit heteroscedastic diagonal noise $\Psi$; probabilistic; handles $n \gg m$.

---

### 3.9 ICA  (`ml_002_05_1`, `ml_002_05_2`)

**Setting:** $x = As$, where $A$ is an unknown mixing matrix and $s$ are independent non-Gaussian sources.

**Goal:** Learn unmixing matrix $W = A^{-1}$.

**Source density:** $p_s(s) = g'(s)$ where $g$ is the sigmoid — a non-Gaussian density.

**Log-likelihood:**
$$\ell(W) = \sum_i \left[\sum_j \log g'(w_j^T x^{(i)}) + \log|\det W|\right]$$

**Gradient update:**
$$W := W + \alpha\left(\begin{bmatrix}1-2g(w_1^Tx^{(i)})\\\vdots\end{bmatrix}x^{(i)T} + (W^T)^{-1}\right)$$

**ICA fails for Gaussian sources:** Rotational symmetry means $A$ and $AR$ are indistinguishable.

## 4. Comparison Table

| Algorithm | Task | Latent variable | Objective | Key assumption |
|---|---|---|---|---|
| **K-means** | Hard clustering | $c^{(i)}\in\{1..k\}$ | Minimise distortion $J$ | Spherical clusters (isotropic distance) |
| **GMM** | Soft clustering / density | $z^{(i)}\sim\text{Cat}(\phi)$ | Maximise marginal $\ell$ via EM | Each cluster is Gaussian |
| **Factor Analysis** | Density / dim reduction | $z\sim\mathcal{N}(0,I)$ | Maximise marginal $\ell$ via EM | Low-rank + diagonal covariance |
| **PCA** | Dim reduction | $y = U_k^T(x-\mu)$ | Maximise projected variance | Linear, orthogonal directions |
| **ICA** | Source separation | $s = Wx$ | Maximise independence (non-Gaussianity) | Sources are independent, non-Gaussian |

---

## 5. Choosing the Right Method

| Situation | Recommended algorithm | Reason |
|---|---|---|
| Group data into $k$ groups, speed matters | **K-means** | Simple, fast coordinate descent |
| Groups have different shapes or sizes | **GMM** | Models covariance per cluster |
| Need a full probability density $p(x)$ | **GMM** | Parametric density estimate |
| $n \gg m$ (more features than samples) | **Factor Analysis** | Handles singular covariance |
| Reduce dimensions, keep most variance | **PCA** | Efficient, closed-form |
| Visualise high-dimensional data | **PCA (k=2 or 3)** | Projects to viewable space |
| Noisy measurement, recover signal | **PCA** | Discard low-eigenvalue noise directions |
| Separate mixed audio / signals | **ICA** | Exploits statistical independence |
| Sources are Gaussian | **ICA fails — use PCA** | Gaussian has rotational symmetry |

## 6. The Role of EM

The **Expectation-Maximisation (EM) algorithm** is the unifying tool for fitting latent variable models:

```
Initialise parameters θ
Repeat until convergence:
    E-step: compute Q_i(z) = p(z | x^(i); θ)           ← posterior over latent z
    M-step: θ := argmax_θ  Σ_i Σ_z Q_i(z) log p(x^(i),z;θ)   ← maximise expected complete-data likelihood
```

**Guarantee:** EM never decreases the log-likelihood $\ell(\theta)$ — each iteration makes progress.

**Connection to K-means:**  
K-means is EM with $Q_i(z) \in \{0,1\}$ (hard assignment) and isotropic equal covariances $\Sigma_j = \sigma^2 I$.

| Model | E-step computes | M-step updates |
|---|---|---|
| GMM | Soft assignments $w_j^{(i)}$ | $\phi_j, \mu_j, \Sigma_j$ |
| Factor Analysis | Gaussian posterior $p(z\mid x)$ | $\mu, \Lambda, \Psi$ |

## 7. Notebook Map (`ml_002_` Series)

| Notebook | Topic | Key content |
|---|---|---|
| `ml_002_00` | **This notebook** — overview | Taxonomy, algorithm survey, comparison |
| `ml_002_01_1` | K-means algorithm | Distortion, coordinate descent, local optima |
| `ml_002_02_1` | Mixture of Gaussians | Soft assignments, density model, EM derivation |
| `ml_002_02_2` | EM for GMM | Full implementation, convergence proof |
| `ml_002_02_3` | Jensen's inequality | Convexity, concavity, ELBO lower bound |
| `ml_002_02_4` | General EM framework | ELBO, coordinate ascent, tightness condition |
| `ml_002_03_1` | Gaussian conditionals | Block Gaussians, Schur complement |
| `ml_002_03_2` | Factor analysis model | Generative story, $\Lambda\Lambda^T + \Psi$, $n\gg m$ |
| `ml_002_03_3` | EM for factor analysis | E-step posterior, M-step, biased vs correct EM |
| `ml_002_04_1` | PCA | Variance maximisation, eigenvectors, eigenfaces |
| `ml_002_04_2` | PCA applications | Compression, visualisation, noise reduction |
| `ml_002_05_1` | ICA model | Cocktail party, density transformation, Gaussian failure |
| `ml_002_05_2` | ICA algorithm | Sigmoid density, stochastic gradient, gradient check |

---

## Practice Questions

**Conceptual**

1. You have 10,000 customer purchase records and want to segment customers into groups. You don't know how many segments to use. Would you start with k-means or GMM? What would you vary, and how would you choose the final number of clusters?

2. A colleague argues: "PCA and Factor Analysis do the same thing — both find a low-dimensional linear subspace." What is the key difference in their assumptions, and give a scenario where this difference matters?

3. You record audio from 3 microphones in a room with 3 speakers. You apply PCA and get 3 components. Are these the original speech signals? If not, what should you use and why?

4. Explain why EM is guaranteed to not decrease the log-likelihood. Which step (E or M) makes progress, and why?

5. The density transformation formula is $p_x(x) = p_s(Wx) \cdot |\det W|$. Intuitively explain why $|\det W|$ appears: what does the determinant measure, and why must the density scale inversely with it?

**True / False — explain**

6. ICA can recover Gaussian sources.

7. K-means is a special case of EM with hard assignments.

8. PCA and Factor Analysis give the same latent representation when $\Psi = \sigma^2 I$ (isotropic noise).

9. The EM algorithm always converges to the global maximum of the likelihood.

10. Adding more components to a GMM always improves test log-likelihood.